# Task 2: test one tier interaction

A targeted CatBoost comparison: original 41 non-noise inputs plus one categorical interaction between loyalty_tier and engagement_level. All other model settings and the five stratified development folds match the original CatBoost. Model selection is based on paired development folds; the 20% holdout has been reused in earlier rounds and is secondary evidence.

In [1]:
from pathlib import Path
from hashlib import sha256
import json,sys,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_data.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
SRC=ROOT/'src'
import numpy as np,pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import f1_score,classification_report,confusion_matrix
from catboost import CatBoostClassifier
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task2.models.decision_rules import restore_categorical_nulls
SEED=set_seed()
DATA='eda_tier_cross_v2'; BASE_DATA='eda_original_v2'
BASE='t2_cat_original_no_noise_v2';NAME='t2_cat_tier_cross_cv5_v5'
dcard=json.loads((SRC/'task2'/'data'/'eda'/f'{DATA}.json').read_text(encoding='utf8'))
base_dcard=json.loads((SRC/'task2'/'data'/'eda'/f'{BASE_DATA}.json').read_text(encoding='utf8'))
bcard=json.loads((SRC/'task2'/'save'/f'{BASE}.json').read_text(encoding='utf8'))
assert dcard['name']==DATA and bcard['data_name']==BASE_DATA
train=pd.read_parquet(SRC/dcard['splits']['train']['path'])
assert len(train)==104000 and train.player_id.is_unique
y=train.segment.astype(int)
raw_X=train.drop(columns=['player_id','segment'])
categorical=raw_X.select_dtypes(include=['object','str']).columns.tolist()
assert len(raw_X.columns)==45 and len(categorical)==11
X=restore_categorical_nulls(raw_X,categorical)
dev_idx,valid_idx=train_test_split(np.arange(len(y)),test_size=.2,stratify=y,random_state=SEED)
X_dev,X_valid=X.iloc[dev_idx],X.iloc[valid_idx]
y_dev,y_valid=y.iloc[dev_idx],y.iloc[valid_idx]
cv_X,cv_y=X_dev.reset_index(drop=True),y_dev.reset_index(drop=True)
splits=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(cv_X,cv_y))
signature=sha256(np.asarray(dev_idx,dtype=np.int64).tobytes()+np.asarray(y_dev,dtype=np.int8).tobytes()).hexdigest()
baseline=joblib.load(SRC/bcard['model_path'])
pre=baseline.named_steps['preprocess']
cat_transform,cat_cols=pre.transformers_[0][1],pre.transformers_[0][2]
num_transform,num_cols=pre.transformers_[1][1],pre.transformers_[1][2]
assert len(cat_cols)==10 and len(num_cols)==31 and len(set(cat_cols+num_cols))==41
assert 'loyalty_engagement_cross' not in cat_cols+num_cols
def make_model():
    new_pre=ColumnTransformer([
        ('categorical',clone(cat_transform),cat_cols+['loyalty_engagement_cross']),
        ('numeric',clone(num_transform),num_cols)],
        remainder='drop',sparse_threshold=0)
    params={**baseline.named_steps['classifier'].get_params(),'cat_features':list(range(11))}
    return Pipeline([('preprocess',new_pre),('classifier',CatBoostClassifier(**params))])
print('Development/holdout:',len(X_dev),len(X_valid))
print('Model input features:',len(cat_cols)+len(num_cols)+1)

Development/holdout: 83200 20800
Model input features: 42


## Paired development folds

Baseline probabilities are loaded from the previously saved OOF cache, with signature checks. Each new fold refits preprocessing on its own training rows; predictions and metadata are cached under a distinct experiment name.

In [2]:
RESULTS=SRC/'task2'/'models'/'results'
base_key={'data_name':BASE_DATA,'data_created_at':base_dcard['created_at'],
          'model_name':BASE,'model_created_at':bcard['created_at'],'seed':SEED,
          'folds':5,'split_signature':signature}
cross_key={'experiment':NAME,'data_name':DATA,'data_created_at':dcard['created_at'],
           'base_key':base_key,'seed':SEED,'folds':5,'split_signature':signature}
base_parts=[];cross_parts=[]
for fold,(fit_pos,check_pos) in enumerate(splits,1):
    bp=RESULTS/f'class_bias_fold{fold}_v1.parquet'
    bm=bp.with_suffix('.json')
    assert json.loads(bm.read_text(encoding='utf8'))=={'key':base_key,'fold':fold}
    old=pd.read_parquet(bp)
    assert old.row_pos.tolist()==check_pos.tolist()
    base_parts.append(old)
    cp=RESULTS/f'tier_cross_fold{fold}_v5.parquet'
    cm=cp.with_suffix('.json')
    if cp.exists() and cm.exists():
        assert json.loads(cm.read_text(encoding='utf8'))=={'key':cross_key,'fold':fold}
        part=pd.read_parquet(cp)
        assert part.row_pos.tolist()==check_pos.tolist()
        print('Loaded fold',fold)
    else:
        assert not (cp.exists() or cm.exists()),'Inspect incomplete cache'
        fitted=make_model()
        fitted.fit(cv_X.iloc[fit_pos],cv_y.iloc[fit_pos])
        prob=np.asarray(fitted.predict_proba(cv_X.iloc[check_pos]),dtype=float)
        part=pd.DataFrame({'fold':fold,'row_pos':check_pos,'true':cv_y.iloc[check_pos].to_numpy(),
            **{f'p_{k}':prob[:,k] for k in range(4)}})
        part.to_parquet(cp,index=False)
        cm.write_text(json.dumps({'key':cross_key,'fold':fold},indent=2)+'\n',encoding='utf8')
        print('Trained fold',fold)
    cross_parts.append(part)
old=pd.concat(base_parts,ignore_index=True);new=pd.concat(cross_parts,ignore_index=True)
assert old.row_pos.tolist()==new.row_pos.tolist() and old.true.tolist()==new.true.tolist()
def labels(frame):return frame[[f'p_{k}' for k in range(4)]].to_numpy().argmax(1)
old_pred=labels(old);new_pred=labels(new)
rows=[]
for fold in range(1,6):
    sel=old.fold==fold
    a=f1_score(old.true[sel],old_pred[sel],average='macro')
    b=f1_score(new.true[sel],new_pred[sel],average='macro')
    rows.append({'fold':fold,'baseline_macro_F1':a,'cross_macro_F1':b,'delta':b-a})
paired=pd.DataFrame(rows)
display(paired)
print('Mean baseline/cross/delta:',paired.baseline_macro_F1.mean(),
      paired.cross_macro_F1.mean(),paired.delta.mean())
print('Fold wins:',int((paired.delta>0).sum()),'/5')
display(pd.DataFrame({'class':['Casual','Grinder','Social','Whale'],
    'baseline_F1':f1_score(old.true,old_pred,average=None,labels=[0,1,2,3]),
    'cross_F1':f1_score(new.true,new_pred,average=None,labels=[0,1,2,3])}))
print('Changed OOF predictions:',int((old_pred!=new_pred).sum()))

Trained fold 1


Trained fold 2


Trained fold 3


Trained fold 4


Trained fold 5


,fold,baseline_macro_F1,cross_macro_F1,delta
0,1,0.756123,0.755800,-0.000323
1,2,0.746062,0.748632,0.002570
2,3,0.750865,0.752726,0.001861
3,4,0.758046,0.755580,-0.002466
4,5,0.751726,0.750497,-0.001229


Mean baseline/cross/delta: 0.7525642419662718 0.7526470322344164 8.279026814468348e-05
Fold wins: 2 /5


,class,baseline_F1,cross_F1
0,Casual,0.821478,0.821770
1,Grinder,0.741199,0.740798
2,Social,0.709997,0.710153
3,Whale,0.737607,0.737882


Changed OOF predictions: 1861


## Fit, save and evaluate the one cross-feature model

The saved model is fitted on the 80% development split. The reused holdout check is reported separately. A full-train refit would be needed before final submission.

In [3]:
SAVE=SRC/'task2'/'save'
model_path,card_path,eval_path=(SAVE/f'{NAME}.joblib',SAVE/f'{NAME}.json',
                                SAVE/f'{NAME}-evaluation.json')
if not any(p.exists() for p in (model_path,card_path,eval_path)):
    model=make_model()
    model.fit(X_dev,y_dev)
    pred=np.asarray(model.predict(X_valid)).reshape(-1).astype(int)
    save_model(task=2,name=NAME,model=model,data_name=DATA,
        card={'idea':'Explicit loyalty_tier x engagement_level categorical cross',
              'base_model':BASE,'seed':SEED,'cv':'five paired development folds',
              'mean_cv_macro_F1':float(paired.cross_macro_F1.mean()),
              'mean_cv_delta':float(paired.delta.mean())})
    result=evaluate_model(task=2,name=NAME,y_true=y_valid,y_pred=pred,
        card={'split':'reused stratified 20% holdout','seed':SEED,
              'selection':'one prespecified cross from OOF error audit'})
else:
    assert all(p.exists() for p in (model_path,card_path,eval_path)),'Inspect incomplete artifact'
    assert json.loads(card_path.read_text(encoding='utf8'))['data_name']==DATA
    model=joblib.load(model_path)
    pred=np.asarray(model.predict(X_valid)).reshape(-1).astype(int)
    result=json.loads(eval_path.read_text(encoding='utf8'))
baseline_pred=np.asarray(baseline.predict(X_valid.drop(columns=['loyalty_engagement_cross']))).reshape(-1).astype(int)
print('Holdout baseline/cross:',f1_score(y_valid,baseline_pred,average='macro'),result['score'])
print('Changed holdout predictions:',int((pred!=baseline_pred).sum()))
print(classification_report(y_valid,pred,digits=4))

Holdout baseline/cross: 0.7504144211991189 0.7495771466435829
Changed holdout predictions: 439
              precision    recall  f1-score   support

           0     0.7787    0.8602    0.8174      8191
           1     0.7484    0.7392    0.7438      5207
           2     0.7426    0.6662    0.7023      4206
           3     0.7814    0.6934    0.7347      3196

    accuracy                         0.7650     20800
   macro avg     0.7628    0.7397    0.7496     20800
weighted avg     0.7642    0.7650    0.7630     20800



## Interpretation

The paired development folds determine whether the explicit cross adds information beyond CatBoost's original splits. The reused holdout is context. Temporal availability of both component tier fields remains unverified.